# AHP
---
_**A**nalytic **H**ierarchy **P**rocess_

In [1]:
import numpy as np
import pandas as pd

In [2]:
import warnings
warnings.filterwarnings("ignore")

## 1. Introduction
---
The **AHP** (_Analytic Hierarchy Process_) method derives priorities through pairwise comparisons and combines them across a hierarchy of a **goal**, **criteria**/**subcriteria**, and **alternatives**.

![](https://upload.wikimedia.org/wikipedia/commons/d/db/AHPHierarchy1Labeled.png)

_Depending on the complexity of the problem, the model may use subcriteria._

### 1.1. Applications
---
The _AHP method_ supports decisions that require prioritizing alternatives, such as:

- Supplier selection
- Assessment of job candidates
- Choice of products
- Feature prioritization

The following example uses randomly generated data for **5 alternatives** and **4 criteria**:

In [3]:
np.random.seed(0)

M = 5
C = np.random.randint(100, 500, M)
df = pd.DataFrame({
    "Criteria 1": C,
    "Criteria 2": (C**(np.random.random(M)/2)).astype(int),
    "Criteria 3": np.log(C).astype(int),
    "Criteria 4": (C*(1 + np.random.random(M))//2).astype(int)
})

df.index = [f'Alternative {i}' for i in np.arange(M)+1]

df.style.background_gradient()

,Criteria 1,Criteria 2,Criteria 3,Criteria 4
Alternative 1,272,11,5,143
Alternative 2,147,8,4,93
Alternative 3,217,5,5,160
Alternative 4,292,2,5,264
Alternative 5,423,2,6,313


## 2. Method steps
---

### 2.1. Pairwise comparison matrix
---
Construct a _pairwise comparison matrix_ $M=[a_{ij}]$ for the $n$ criteria, where $a_{ij}>0$ expresses the importance of criterion $i$ relative to criterion $j$. Use the following **scale of relative importance**:

|       scale        | importance             |
| :----------------: | :--------------------- |
|         1          | Equal importance       |
|         3          | Moderate importance    |
|         5          | Strong importance      |
|         7          | Very strong importance |
|         9          | Extreme importance     |
|     2, 4, 6, 8     | Intermediate values    |
|   1/2, ..., 1/9    | Reciprocal values      |

Set $a_{ii}=1$ and enforce **reciprocity**, $a_{ji}=1/a_{ij}$. For three criteria **A**, **B**, and **C**, the matrix is:

$$ \large
M =
\begin{bmatrix}
    1 & A_B & A_C \\
    B_A & 1 & B_C \\
    C_A & C_B & 1
\end{bmatrix} =
\begin{bmatrix}
    1 & A_B & A_C \\
    \frac{1}{A_B} & 1 & B_C \\
    \frac{1}{A_C} & \frac{1}{B_C} & 1
\end{bmatrix}
$$

Perfect **consistency** additionally requires $a_{ij}a_{jk}=a_{ik}$ for every triplet of criteria. The consistency ratio assesses departures from this condition.

In [4]:
PWCM = pd.DataFrame([
    [1  , 5  , 4  , 7],
    [1/5, 1  , 1/2, 3],
    [1/4, 2  , 1  , 3],
    [1/7, 1/3, 1/3, 1]
]).astype(float)

PWCM.index = df.columns
PWCM.columns = df.columns

PWCM

,Criteria 1,Criteria 2,Criteria 3,Criteria 4
Criteria 1,1.000000,5.000000,4.000000,7.0
Criteria 2,0.200000,1.000000,0.500000,3.0
Criteria 3,0.250000,2.000000,1.000000,3.0
Criteria 4,0.142857,0.333333,0.333333,1.0


### 2.2. Principal eigenvalue and eigenvector
---
Calculate the **principal eigenvalue** $\lambda_{\max}$, the matrix's largest positive eigenvalue, and its corresponding right eigenvector $v$:

$$ \large
Mv=\lambda_{\max}v
$$

In [5]:
eigenvalues, eigenvectors = np.linalg.eig(
    PWCM.to_numpy(dtype=float)
)

principal_idx = np.argmax(eigenvalues.real)

lambda_max = float(eigenvalues[principal_idx].real)
priority_vector = eigenvectors[:, principal_idx].real

lambda_max, priority_vector

(4.099205194317488, array([0.92797281, 0.20143921, 0.29828211, 0.09652195]))

### 2.3. Criteria weights
---
Normalize the principal eigenvector to obtain **criteria weights** that sum to one:

$$ \large
w_i=\frac{v_i}{\sum_{j=1}^{n}v_j}
$$

In [6]:
CRITERIA_WEIGHTS = pd.Series(
    priority_vector / priority_vector.sum(),
    index=PWCM.index
)

CRITERIA_WEIGHTS

Criteria 1    0.608820
Criteria 2    0.132159
Criteria 3    0.195695
Criteria 4    0.063326
dtype: float64

### 2.4. Consistency ratio
---
Assess the consistency of the pairwise judgments using the **consistency index**:

$$ \large
CI = \frac{\lambda_{\max} - n}{n - 1}
$$

The **consistency ratio** compares this index with the random consistency index:

$$ \large
CR = \frac{CI}{RI}
$$

$RI$ is the mean $CI$ of randomly generated reciprocal matrices of the same order $n$, using the same judgment scale.

| order |  1  |  2  |  3   |  4  |  5   |  6   |  7   |  8   |  9   |  10  |
| :---: | :-: | :-: | :--: | :-: | :--: | :--: | :--: | :--: | :--: | :--: |
|  RI   |  0  |  0  | 0.58 | 0.9 | 1.12 | 1.24 | 1.32 | 1.41 | 1.45 | 1.49 |

A $CR$ **below 0.10** is commonly considered acceptable. Larger values suggest reviewing the pairwise judgments.

In [7]:
n = len(CRITERIA_WEIGHTS)
CI = (lambda_max - n)/(n - 1)

CI

0.03306839810582929

In [8]:
RI = {
    1: 0,
    2: 0,
    3: 0.58,
    4: 0.90,
    5: 1.12,
    6: 1.24,
    7: 1.32,
    8: 1.41,
    9: 1.45,
    10: 1.49
}[n]

CR = CI/RI

print(f'''
Consistency ratio: {CR:.04f}
    Less than 10%: {CR < 0.1}
''')


Consistency ratio: 0.0367
    Less than 10%: True



### 2.5. Score and Ranking
---
After checking judgment consistency, combine criterion weights with the alternatives' **local priorities**. This example assumes that preference is proportional to positive benefit values and inversely proportional to positive cost values.

Let $x_{ij}>0$ be the performance of alternative $i$ on criterion $j$, with $m$ alternatives. Transform the performances according to each criterion's objective:

$$ \large
z_{ij} =
\begin{cases}
    x_{ij}, & \text{if criterion } j \text{ is maximized}, \\
    \frac{1}{x_{ij}}, & \text{if criterion } j \text{ is minimized}.
\end{cases}
$$

Normalize each criterion by its column sum and compute the **score**:

$$ \large
S_i = \sum_{j=1}^{n} w_j \frac{z_{ij}}{\sum_{k=1}^{m} z_{kj}}
$$

Rank alternatives by **decreasing score**.

In [9]:
df_a = df.copy()

df_a["Criteria 2"] = 1/df_a["Criteria 2"]
df_a = df_a/df_a.sum()

df["score"] = np.sum(df_a*CRITERIA_WEIGHTS, axis=1)
df["ranking"] = (
    df["score"]
    .rank(ascending=False, method="first")
    .astype(int)
)

(
    df.sort_values("ranking")
    .style.background_gradient(
        subset=list(df_a.columns)
    )
)

,Criteria 1,Criteria 2,Criteria 3,Criteria 4,score,ranking
Alternative 5,423,2,6,313,0.304630,1
Alternative 4,292,2,5,264,0.234578,2
Alternative 1,272,11,5,143,0.179506,3
Alternative 3,217,5,5,160,0.166010,4
Alternative 2,147,8,4,93,0.115276,5


## 3. Examples
---

### 3.1. Device prioritization
---
Prioritize devices using **4 criteria**: **price**, **camera**, **display**, and **battery**. The data comes from [DXOMARK](https://www.dxomark.com/smartphones/). **Price** is minimized; the remaining criteria are maximized.

In [10]:
df_ex1 = pd.DataFrame({
    # criteria we want to minimize
    "price": [1199, 999, 1299, 1199, 908],
    # criteria we want to maximize
    "camera": [154, 153, 149, 140, 140],
    "display": [151, 154, 133, 148, 130],
    "battery": [134, 111, 103, 142, 133]
})

df_ex1.index = [
    "Apple iPhone 15 Pro Max", 
    "Google Pixel 8 Pro",
    "Huawei Mate 50 Pro",
    "Samsung Galaxy S23 Ultra",
    "Xiaomi 13 Ultra",
]

df_ex1.style.background_gradient()

,price,camera,display,battery
Apple iPhone 15 Pro Max,1199,154,151,134
Google Pixel 8 Pro,999,153,154,111
Huawei Mate 50 Pro,1299,149,133,103
Samsung Galaxy S23 Ultra,1199,140,148,142
Xiaomi 13 Ultra,908,140,130,133


In [11]:
# Pairwise comparison matrix
# price > camera > display > battery
PWCM_ex1 = pd.DataFrame([
    [1  , 3  , 7  , 5],
    [1/3, 1  , 4  , 2],
    [1/7, 1/4, 1  , 2],
    [1/5, 1/2, 1/2, 1]
]).astype(float)

PWCM_ex1.index = df_ex1.columns
PWCM_ex1.columns = df_ex1.columns

# Principal eigenvalue and eigenvector
eigenvalues, eigenvectors = np.linalg.eig(
    PWCM_ex1.to_numpy(dtype=float)
)

principal_idx = np.argmax(eigenvalues.real)

lambda_max = float(eigenvalues[principal_idx].real)
priority_vector = eigenvectors[:, principal_idx].real

# Criteria weights
CRITERIA_WEIGHTS = pd.Series(
    priority_vector / priority_vector.sum(),
    index=PWCM_ex1.index
)

# Consistency ratio
n = len(CRITERIA_WEIGHTS)
CI = (lambda_max - n) / (n - 1)

if not CR < 0.1:
    raise Exception("Consistency ratio not acceptable!")

# Application
df_ex1_a = df_ex1.copy()

df_ex1_a["price"] = 1/df_ex1_a["price"]
df_ex1_a = df_ex1_a/df_ex1_a.sum()

df_ex1["score"] = np.sum(df_ex1_a*CRITERIA_WEIGHTS, axis=1)
df_ex1["ranking"] = (
    df_ex1["score"]
    .rank(ascending=False, method="first")
    .astype(int)
)

(
    df_ex1.sort_values("ranking")
    .style.background_gradient(
        subset=list(df_ex1_a.columns)
    )
)

,price,camera,display,battery,score,ranking
Xiaomi 13 Ultra,908,140,130,133,0.221517,1
Google Pixel 8 Pro,999,153,154,111,0.213215,2
Apple iPhone 15 Pro Max,1199,154,151,134,0.195233,3
Samsung Galaxy S23 Ultra,1199,140,148,142,0.191427,4
Huawei Mate 50 Pro,1299,149,133,103,0.178608,5


## 4. References
---

- Saaty, T. L. (2008). [*Decision making with the analytic hierarchy process*](https://doi.org/10.1504/IJSSCI.2008.017590).
- Saaty, R. W. (1987). [*The analytic hierarchy process: what it is and how it is used*](https://doi.org/10.1016/0270-0255(87)90473-8).